# 4 — Complete all 3,456 physical episodes
Launch shared ID first. Only after all 1,152 ID artifacts validate should OOD start. Never run both workers concurrently. All launchers use deterministic manifest-backed resume.


In [ ]:
import csv, os, subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; OUT=Path.home()/"stage3_new"; P=Path.home()/"LIBERO-plus"; N=Path.home()/"stage1-native"; GPU=(Path.home()/"stage3_new_gpu.txt").read_text().strip()
def alive(pidfile):
    if not pidfile.exists(): return False
    try: os.kill(int(pidfile.read_text()),0); return True
    except (ProcessLookupError,PermissionError,ValueError): return False
def launch(scene):
    other=OUT/f"stage3_new_{'ood' if scene=='id' else 'id'}.pid"
    if alive(other): raise SystemExit(f"STOP: opposite-scene worker {other.read_text().strip()} is alive")
    pidfile=OUT/f"stage3_new_{scene}.pid"; log=OUT/f"stage3_new_{scene}.log"
    if alive(pidfile): raise SystemExit(f"STOP: {scene} worker {pidfile.read_text().strip()} is already alive")
    py=Path.home()/("venv-stage1-id/bin/python" if scene=="id" else "venv-stage1-ood/bin/python")
    env=os.environ.copy(); env.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg","PYTHONUNBUFFERED":"1"})
    if scene=="ood": env.update({"PYTHONPATH":str(P),"MAGICK_HOME":str(N),"PATH":str(N/"bin")+os.pathsep+env.get("PATH",""),"LD_LIBRARY_PATH":str(N/"lib")+os.pathsep+env.get("LD_LIBRARY_PATH","")})
    cmd=[str(py),"-u","-m","async_vla_benchmark.scripts.run_stage3_new","--config",str(R/"async_vla_benchmark/configs/stage3_new.yaml"),"--manifest",str(OUT/"stage3_new_manifest.csv"),"--output-dir",str(OUT),"--scene",scene,"--resume","--verbose"]
    fh=open(log,"ab"); proc=subprocess.Popen(cmd,cwd=R,env=env,stdout=fh,stderr=subprocess.STDOUT,start_new_session=True); pidfile.write_text(str(proc.pid)+"\n"); print("launched",scene,proc.pid,log)
launch("id")


In [ ]:
# Safe to rerun after reconnecting: progress comes from valid durable triplets, not the CSV alone.
manifest=list(csv.DictReader(open(OUT/"stage3_new_manifest.csv"))); planned={r['run_id'] for r in manifest}
done={p.stem for p in (OUT/"episodes").glob("*.json")} & {p.stem for p in (OUT/"requests").glob("*.parquet")} & {p.stem for p in (OUT/"actions").glob("*.parquet")}
print(f"physical artifacts {len(planned & done)}/3456; remaining {3456-len(planned & done)}")
for scene in ("id","ood"):
    pf=OUT/f"stage3_new_{scene}.pid"; log=OUT/f"stage3_new_{scene}.log"; print(scene,"alive=",alive(pf)); print(''.join(log.read_text(errors='replace').splitlines(True)[-10:]) if log.exists() else '(not started)')
print("Checkpoint ~/stage3_new off-machine regularly.")


In [ ]:
# Run only after the ID worker exits and all 1,152 shared-ID triplets exist.
rows=list(csv.DictReader(open(OUT/"stage3_new_manifest.csv"))); expected={r['run_id'] for r in rows if r['scene']=='id'}
done={p.stem for p in (OUT/"episodes").glob("*.json")} & {p.stem for p in (OUT/"requests").glob("*.parquet")} & {p.stem for p in (OUT/"actions").glob("*.parquet")}
if alive(OUT/"stage3_new_id.pid") or len(expected & done)!=1152: raise SystemExit(f"STOP: ID is {len(expected & done)}/1152 or worker remains alive")
subprocess.run([str(Path.home()/"venv-stage1-id/bin/python"),"-m","async_vla_benchmark.scripts.validate_stage3_new","--manifest",str(OUT/"stage3_new_manifest.csv"),"--output-dir",str(OUT),"--require-scene","id"],cwd=R,check=True)
launch("ood")
